## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 160.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 265.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 170.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 231.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.
opentelemetry-exporter-otlp

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub pandas tiktoken pymupdf langchain langchain-community chromadb sentence-transformers langchain-huggingface  numpy -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
# Defining the Hugging Face repository and model version for Mistral-7B
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
# Specifying the file name for the quantized Mistral-7B model
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [5]:
# Downloading the specified model file from Hugging Face Hub and store its local path
model_path = hf_hub_download(
    repo_id= model_name_or_path, #Complete the code to mention the repo id
    filename= model_basename #Complete the code to mention the model name
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
# Loading the LLaMA model with specified context, GPU layers, and batch size
llm = Llama(
    model_path=model_path,
    n_ctx=5300,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 1 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [7]:
def response(query,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
   # Sends the query to the LLM with specified parameters
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    # Extracting and returning only the text part of the response
    return model_output['choices'][0]['text']

In [ ]:
response("What treatment options are available for managing stomach pain?")

"\n\nStomach pain, also known as abdominal pain, can have various causes and may require different treatments. Here are some common treatment options for managing stomach pain:\n\n1. Medications: Over-the-counter (OTC) or prescription medications may be recommended to help alleviate stomach pain. These include antacids for heartburn and acid reflux, anti-diarrheal agents for diarrhea, antibiotics for bacterial infections, and pain relievers such as acetaminophen or nonsteroidal anti-inflammatory drugs (NSAIDs) for inflammation or pain.\n2. Dietary modifications: Making dietary changes can help manage stomach pain caused by conditions like irritable bowel syndrome (IBS), lactose intolerance, or food allergies. Avoiding trigger foods and eating smaller, more frequent meals may help reduce symptoms.\n3. Probiotics: Probiotics are live bacteria and yeasts that can help restore the balance of good bacteria in the gut, which can alleviate stomach pain caused by conditions like IBS or antibio

####Observations


*   The response generated by the LLM is accurate and medically appropriate for managing stomach pain



### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [9]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clin

####Observations


*   The response generated by the LLM is accurate and medically appropriate for managing sepsis in critical care unit


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [10]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


"\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.\n3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the early stages.\n4. Fever: A fever may be present, particularly if the appendix has ruptured or perforated.\n5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.\n6. Abdominal swelling: The abdomen may become swollen and tender to the touch.\n7. Inability to pass gas: Passing gas can be difficult due to the infla

####Observations


*   The response generated by the LLM is accurate and medically appropriate. It accurately identifies the symptoms


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [11]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, or eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.\n\nThere are several treatments that have been shown to be effective in addressing sudden patchy hair loss:\n\n1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.\n2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by increa

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [12]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a brain injury, such as pain, swelling, seizures, or infections.\n3. Surgery: In some cases, surgery may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.\n4. Rehabilitation: Rehabilitation is an essential component of treatment for brain injuries. It may include physical therapy, occupational therapy, speech and langua

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [13]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.\n4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.\n5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, seek medical help as soon as possible.\n\nOnce you've ensured the person's safety and stabi

## Question Answering using LLM with Prompt Engineering

In [14]:
system_prompt = """
You are an AI Assistant who has expertise in medical domain. You are expected to be clear to the point and provide relevant response based on the context.
Your responses should be accepted by medical professional or medical standards.
If you do not know the answer, please say "I don't know" and avoid giving any generic response.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [15]:
input1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(input1)

Llama.generate: prefix-match hit


'\n\nThe protocol for managing sepsis in a critical care unit involves early recognition, rapid resuscitation, source control, antibiotic therapy, and close monitoring. Here are the steps:\n\n1. Early Recognition: Identify sepsis suspects based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidosis, tachycardia, hypotension), and organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score.\n\n2. Rapid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore circulating volume. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider using vasopressors if necessary to maintain adequate blood pressure.\n\n3. Source Control: Identify and address the source of infection, which may include surgical intervention or drainage procedures.\n\n4. Antibiotic Therapy: Administer broad-spectrum antibiotics based on suspected pathogens 

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [16]:
input2 = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(input2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms of appendicitis include:\n\n1. Sudden pain in the lower right abdomen, which may be mild at first but becomes sharp and severe over time.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, which may start after the abdominal pain begins.\n5. Constipation or diarrhea.\n6. Abdominal swelling and tenderness.\n7. Inability to pass gas or have a bowel movement.\n8. Feeling restless or unable to find a comfortable position due to pain.\n\nAppendicitis cannot be cured via medicine alone, as the inflammation can lead to the appendix rupturing if left untreated. If the appendix ruptures, it can lead to peritonitis, a serious infection of the abdominal cavity that requires immediate surgical intervention. The standard treatment for appendicitis is an appendectomy, which i

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [17]:
input3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(input3)

Llama.generate: prefix-match hit


"\n\nPossible causes of sudden patchy hair loss, also known as alopecia areata, include:\n1. Autoimmune disorders: In this condition, the body's immune system attacks the hair follicles, leading to hair loss.\n2. Stress: Emotional or physical stress can trigger hair loss.\n3. Vitamin deficiencies: Lack of certain vitamins like iron, zinc, and biotin can cause hair loss.\n4. Hormonal imbalances: Thyroid disorders, polycystic ovary syndrome (PCOS), and other hormonal imbalances can lead to hair loss.\n5. Medications: Certain medications like chemotherapy drugs, antidepressants, and beta-blockers can cause hair loss as a side effect.\n6. Infections: Fungal infections like ringworm or bacterial infections can cause patchy hair loss.\n7. Trauma: Physical trauma to the scalp, such as burns or injuries, can lead to hair loss.\n\nEffective treatments for addressing sudden patchy hair loss include:\n1. Corticosteroids: Topical or injected corticosteroids can help reduce inflammation and promote

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [18]:
input4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(input4)

Llama.generate: prefix-match hit


"\n\nA person with a brain injury may require a combination of medical interventions and rehabilitative therapies depending on the severity and location of the injury. Here are some common treatments:\n\n1. Emergency care: For severe brain injuries, immediate medical attention is necessary to prevent further damage or complications. This may include surgery to remove hematomas or decompressing skull fractures, administering medications to control swelling or seizures, and providing supportive care such as maintaining breathing and blood pressure.\n2. Medications: Depending on the symptoms, various medications may be prescribed to manage conditions such as pain, seizures, infections, and depression. For instance, anti-inflammatory drugs can help reduce swelling, while anticonvulsants are used to prevent seizures.\n3. Rehabilitative therapies: These therapies aim to help the person regain lost skills or compensate for impairments. Some common rehabilitative therapies include physical the

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [19]:
input5 = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(input5)

Llama.generate: prefix-match hit


"\n\nA fractured leg, specifically a tibia or fibula fracture, is a common injury that can occur during hiking due to falls, tripping over rocks, or other traumatic events. Here are the necessary precautions and treatment steps for such an individual:\n\n1. Assess the severity of the injury: Check for signs of open wounds, deformities, swelling, numbness, or inability to move the leg. If there is significant bleeding, apply a clean cloth to control it and elevate the leg to reduce swelling. Do not attempt to realign the bone if it is visibly misaligned.\n2. Immobilize the leg: Use a splint, sling, or a makeshift immobilizer made from available materials (such as sticks, clothing, or backpack) to prevent further movement and potential damage to the fracture. Be sure to keep the injured leg elevated above heart level to minimize swelling and pain.\n3. Seek medical attention: It is essential to seek professional medical help as soon as possible. Fractures can lead to complications such as

## Data Preparation for RAG

In [20]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

### Loading the Data

In [21]:
from google.colab import drive
drive.mount('/content/drive')
manual_pdf_path = "/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Data Overview

#### Checking the first 5 pages

In [22]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
uday.desiraju@gmail.com
X4SYCGN7EW
This file is meant for personal use by uday.desiraju@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
uday.desiraju@gmail.com
X4SYCGN7EW
This file is meant for personal use by uday.desiraju@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

#### Checking the number of pages

In [23]:
len(manual)

4114

### Data Chunking

In [24]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512, #Complete the code to define the chunk size
    chunk_overlap= 20 #Complete the code to define the chunk overlap
)

In [25]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [26]:
len(document_chunks)

8469

In [27]:
document_chunks[0].page_content

'uday.desiraju@gmail.com\nX4SYCGN7EW\nThis file is meant for personal use by uday.desiraju@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [28]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

In [29]:
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nuday.desiraju@gmail.com\nX4SYCGN7EW\nThis file is meant for personal use by uday.desiraju@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

### Embedding

In [30]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

/tmp/ipykernel_48412/3102610685.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")


In [31]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [32]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [33]:
embedding_1,embedding_2

([-0.01110038161277771,
  -0.007869027554988861,
  0.003941408358514309,
  -0.016412794589996338,
  -0.015166081488132477,
  -0.02910950593650341,
  -0.018975110724568367,
  0.03810461610555649,
  0.01881696842610836,
  0.041289072483778,
  0.025547528639435768,
  0.001908413483761251,
  0.027581699192523956,
  -0.02450316585600376,
  -0.004729866050183773,
  -0.0038564405404031277,
  0.003108871867880225,
  -0.024181460961699486,
  0.0011145883472636342,
  0.012282798066735268,
  0.011673943139612675,
  0.01889336109161377,
  -0.06530880928039551,
  -0.03919079527258873,
  0.005695889238268137,
  0.037697553634643555,
  -0.00032704617478884757,
  -0.022029263898730278,
  0.054199833422899246,
  0.06140422448515892,
  -0.01838448829948902,
  -0.005174989812076092,
  0.021416377276182175,
  -0.04018021374940872,
  -0.02268550544977188,
  -0.007573512848466635,
  0.05773623660206795,
  -0.03268420323729515,
  -0.023395901545882225,
  -0.0422658734023571,
  0.009391671046614647,
  -0.0230

### Vector Database

In [34]:
out_dir = 'medicalinfo_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [35]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [36]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_48412/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [37]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [38]:
vectorstore.similarity_search("how to treat covid?",k=2)

[Document(metadata={'moddate': '2026-04-25T02:36:53+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationDate': 'D:20120615054440Z', 'total_pages': 4114, 'file_path': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'author': '', 'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'trapped': '', 'source': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'page': 1573, 'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'modDate': 'D:20260425023653Z', 'keywords': ''}, page_content='antiviral treatment in these patients appears to reduce the incidence of lower respiratory disease and\nhospitalization. Appropriate antibacterial therapy decreases the mortality rate due to secondary bacterial\npneumonia.\nTreatment\n• Symptomatic treatment\n• Sometimes antiviral drugs\nTreatment for most patients is symptomatic, including rest, hydration, and antipyretics

### Retriever

In [39]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [40]:
rel_docs = retriever.get_relevant_documents("What are treatments for common cold?")
rel_docs

/tmp/ipykernel_48412/1441040601.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What are treatments for common cold?")


[Document(metadata={'moddate': '2026-04-25T02:36:53+00:00', 'creationDate': 'D:20120615054440Z', 'page': 1570, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'format': 'PDF 1.7', 'trapped': '', 'total_pages': 4114, 'modDate': 'D:20260425023653Z', 'source': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'keywords': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'author': ''}, page_content='ipratropium bromide (2 sprays of a 0.03% solution bid or tid); however, these drugs should be avoided in\nthe elderly and people with benign prostatic hypertrophy or glaucoma. First-generation antihistamines\nfrequently cause sedation, but 2nd-generation (nonsedating) antihistamines are ineffective for treating the\ncommon cold. Antihistamines and decongestants are not reco

### System and User Prompt Template

In [41]:
qna_system_message = """
You are a medical knowledge assistant designed to support healthcare professionals by providing accurate, reliable, and concise medical information.

Your task is to answer user questions using ONLY the information provided in the context, which is sourced from trusted medical references such as the Merck Manuals.

The context will begin with the token: ###Context
The user question will begin with the token: ###Question

Rules you must follow:
- Use only the information explicitly stated in the provided context.
- Do NOT add external medical knowledge, assumptions, or personal opinions.
- Do NOT mention or refer to the context, documents, or source materials in your final answer.
- If the answer is not available or cannot be clearly derived from the context, respond exactly with:
  "I don't know"

Your responses should be:
- Clinically accurate
- Clear and concise
- Structured where appropriate (bullet points or steps)
- Written in professional medical language suitable for healthcare practitioners
"""

In [42]:
qna_user_message_template = """
###Context
The following excerpts are retrieved from medical reference documents relevant to the question below.

{context}

###Question
{question}
"""

### Response Function

In [43]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

> Add blockquote



In [44]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,top_k=20)

Llama.generate: prefix-match hit


"Based on the context provided, I cannot clearly derive a specific protocol for managing sepsis in a critical care unit. The context discusses various aspects of critical care medicine, including patient monitoring and testing, supportive care, and blood tests. However, it does not provide detailed information about the management of sepsis. Therefore, my answer is:\n\nI don't know"

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [45]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,top_k=20)

Llama.generate: prefix-match hit


"Based on the context provided:\n\nCommon symptoms for appendicitis include pain in the lower right abdomen, loss of appetite, nausea or vomiting, fever, and a feeling of being sick. Appendicitis cannot be cured via medicine alone; surgical removal of the appendix (appendectomy) is required to treat it. The surgery should be preceded by IV antibiotics, and third-generation cephalosporins are preferred. If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [46]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,top_k=20)

Llama.generate: prefix-match hit


'Based on the provided context, the condition described as "sudden patchy hair loss in people with no obvious skin or systemic disorder" is referred to as Alopecia Areata. The context suggests that this condition may be an autoimmune disorder and that corticosteroids, retinoids, or immunosuppressants could be effective treatments. However, the context does not provide information on the causes of Alopecia Areata beyond being an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.\n\nTherefore, the answer would be:'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [47]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided:\n- Initial treatment consists of ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors, decompress the brain, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n- Structural changes from head injury may be gross or microscopic, depending on the mechanism and forces involved. Clinical manifestations'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [48]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,top_k=20)

Llama.generate: prefix-match hit


"Based on the provided context, here's the answer:\n\n1. Elevate the injured limb above heart level for the first 2 days to minimize swelling and aid in draining edema fluid.\n2. After 48 hours, apply heat (using a heating pad) for 15-20 minutes to relieve pain and potentially speed up healing.\n3. Immobilize the injured limb and joints proximal and distal to prevent further injury and facilitate healing. A cast may be used for fractures or injuries requiring weeks of immobilization.\n4."

### Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?




In [49]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,temperature=0.5)

Llama.generate: prefix-match hit


"Based on the provided context, I don't have enough information to answer the question directly. However, I can provide some general information about critical care units and patient monitoring that may be relevant to managing sepsis.\n\nCritical care units specialize in caring for seriously ill patients, who are best treated by experienced personnel. ICUs have a high nurse:patient ratio to ensure adequate treatment and monitoring of physiologic parameters. Supportive care includes provision of adequate nutrition, prevention of infection, stress ulcers and gastritis, and pulmonary embolism.\n\nPatient monitoring in the IC"

###Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


In [50]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,temperature=0.5, max_tokens =128)

Llama.generate: prefix-match hit


"Based on the context, the symptoms of appendicitis include pain, particularly in the lower right abdomen. The treatment for appendicitis is surgical removal of the appendix (appendectomy), which may be performed open or laparoscopically. Antibiotics are used to treat any associated infection before and/or after surgery. If the appendix has perforated, antibiotics should be continued until the patient's temperature and white blood cell count have normalized. In cases where a large inflammatory mass is found involving the appendix, terminal ileum, and cecum"

###Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [51]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,top_p=0.9, top_k=30)

Llama.generate: prefix-match hit


'Based on the provided context, the condition described as "sudden patchy hair loss in people with no obvious skin or systemic disorder" is referred to as Alopecia Areata. The context suggests that this condition may be an autoimmune disorder and that corticosteroids, retinoids, or immunosuppressants could be effective treatments. However, the context does not provide information on the causes of Alopecia Areata beyond being an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.\n\nTherefore, the answer would be:'

###Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [52]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,top_p=0.9, top_k=10)

Llama.generate: prefix-match hit


'Based on the context provided:\n- Initial treatment consists of ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors, decompress the brain, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n- Structural changes from head injury may be gross or microscopic, depending on the mechanism and forces involved. Clinical manifestations'

###Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [53]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,top_p= 0.95, top_k=10)

Llama.generate: prefix-match hit


"Based on the provided context, here's the answer:\n\n1. Elevate the injured limb above heart level for the first 2 days to minimize swelling and aid in draining edema fluid.\n2. After 48 hours, apply heat (using a heating pad) for 15-20 minutes to relieve pain and potentially speed up healing.\n3. Immobilize the injured limb and joints proximal and distal to prevent further injury and facilitate healing. A cast may be used for fractures or injuries requiring weeks of immobilization.\n4."

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [54]:
groundedness_rater_system_message  = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [55]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [56]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [57]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [58]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input_1,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify if the answer is derived only from the information presented in the context.
2. Check if there is any explicit information about the protocol for managing sepsis in the context.
3. Determine if the answer accurately reflects the absence of information in the context regarding the protocol for managing sepsis.

Explanation:
The answer states that there is no explicit information about the protocol for managing sepsis in the context. This is an accurate reflection of the context as there is indeed no mention of a specific protocol for managing sepsis in the provided text. Therefore, the answer is derived only from the information presented in the context and adheres to the metric.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the answer as a 5 because it follows the metric completely by deriving the answer solely from the context provided.

 To evaluate the context as per the relevance me

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [64]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input_2,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the common symptoms mentioned in the context for appendicitis.
2. Check if the answer correctly lists those symptoms.
3. Determine if the answer states that appendicitis cannot be cured via medicine and requires surgical removal.
4. Verify if the surgical procedure mentioned in the answer (open or laparoscopic appendectomy) is consistent with the context.
5. Ensure that the answer does not include any information not present in the context.

The answer adheres to the metric as it lists the common symptoms for appendicitis directly from the context and states that appendicitis cannot be cured via medicine and requires surgical removal, which is consistent with the context. Therefore, the metric is followed completely.

Rating: 5 (The metric is followed completely)

 Steps to evaluate context as per relevance metric:
1. Identify the main aspects of the question: common symptoms for appendicitis and whether it can be cured via medicine or if surg

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [60]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input_3,max_tokens=350)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the question and the specific information being asked for in the question.
2. Read through the context carefully to understand the key points related to the question.
3. Determine if the AI generated answer is derived only from the information presented in the context.

Explanation:
The question asks about effective treatments or solutions for sudden patchy hair loss and possible causes behind it. The context provides information on Alopecia Areata, which is a condition characterized by sudden patchy hair loss with no obvious skin or systemic disorder. The context also mentions that this condition is thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. The AI generated answer correctly identifies Alopecia Areata as the condition causing sudden patchy hair loss and lists corticosteroids, retinoids, or immunosuppressants as effective treatments. It also correctly states that the

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [61]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input_4,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information in the context related to treatments for TBI.
2. Determine if the AI generated answer is derived only from the information presented in the context.

Explanation:
The AI generated answer summarizes the treatments mentioned in the context, which includes ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, as well as possible surgery for more severe injuries to place monitors, decompress the brain, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are also important. The answer is derived directly from the context, as it only repeats the information provided in the context without adding any new details or interpretations.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the answer a 5, as the metric is f

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [63]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input_5,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to precautions and treatment steps for a person who has fractured their leg.
2. Compare each point in the answer with the corresponding information in the context to ensure that the answer is derived only from the context.
3. Evaluate the extent to which the metric is followed.

Explanation:
The answer adheres to the metric as it includes all the necessary precautions and treatment steps for a person who has fractured their leg, which are directly derived from the information in the context. The answer does not include any additional or irrelevant information.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the answer as 5 (The metric is followed completely).

 To evaluate the context as per the relevance metric:
1. Identify the main aspects of the question: necessary precautions and treatment steps for a person who has fractured their leg during a h

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___